<div align="center">

# YAPAY ZEKÂ DESTEKLİ İLAÇ TASARIMI

**Erzincan Binali Yıldırım Üniversitesi — Sürekli Eğitim Uygulama ve Araştırma Merkezi**

28–29 Kasım 2026 · ML tabanlı QSAR uygulamalı eğitimi

</div>

---

### Eğitim sorumlusu

**Dr. Öğr. Üyesi Ekrem Yaşar**
Erzincan Binali Yıldırım Üniversitesi, Tıp Fakültesi, Biyofizik Anabilim Dalı

📧 `ey.biophysics@gmail.com`

---

# Oturum 3 — Regresyon: Aktiviteyi Sayı Olarak Tahmin Etmek

**Girdi:** Oturum 2'nin üç özellik seti
**Çıktı:** eğitilmiş model, sonuç tablosu, eğitim/test bölünmesi

Artık sayı matrislerimiz var. Bu oturumda bir molekülün `pIC50` değerini
**sürekli bir sayı** olarak tahmin edeceğiz.

## Oturumun asıl konusu: dürüst değerlendirme

Model eğitmek kolaydır — `model.fit(X, y)` tek satırdır. Zor olan, modelin
**gerçekten işe yarayıp yaramadığını** anlamaktır. QSAR literatüründe
yayınlanmış pek çok model olduğundan iyi görünür; çünkü değerlendirme protokolü
kusurludur.

Oturumun iskeleti şöyle:

```
                 ┌──────────── EĞİTİM (%80) ───────────┐  ┌─ TEST (%20) ─┐
1. Bölünme       │                                     │  │   DOKUNMA    │
2. Tarama        │  iç-eğitim      │  doğrulama        │  │   DOKUNMA    │
3. Optimizasyon  │  5 katlı çapraz doğrulama           │  │   DOKUNMA    │
4. Son eğitim    │  bütün eğitim verisi                │  │              │
5. Değerlendirme │                                     │  │  TEK SEFER   │
                 └─────────────────────────────────────┘  └──────────────┘
6. y-randomizasyon: modelin öğrendiğini doğrula
```

> **Altın kural:** Test seti, model seçimi ve ayarlaması tamamen bittikten
> sonra **yalnızca bir kez** kullanılır. Test setine bakıp model
> değiştirirseniz o artık bir test seti değildir ve bildirdiğiniz performans
> iyimser olur.

## Bu oturumda yapacaklarımız

| Adım | Araç | Amaç |
|---|---|---|
| Eğitim/test bölünmesi | rastgele **ve** iskelet bazlı | Gerçekçi değerlendirme |
| Hızlı model taraması | **LazyPredict** (~40 model) | Hangi aile umut verici? |
| Hiperparametre optimizasyonu | **Optuna** | En iyi ayarları bulmak |
| Çapraz doğrulama | 5 katlı K-Fold | Kararlılığı ölçmek |
| Son değerlendirme | tutulan test seti | Dürüst performans |
| **y-randomizasyon** | hedefi karıştırmak | Model gerçekten öğrendi mi? |

⏱️ Yaklaşık süre: 15–25 dakika (kursun en uzun defteri)

---
## 1. Ayarlar

| Ayar | Ne işe yarar? |
|---|---|
| `OZELLIK_SETI` | Hangi temsille çalışacağız. Tarama üçünde de yapılır, ama optimizasyon ve son model tek set üzerinde kurulur. |
| `OPTUNA_DENEME` | Optuna kaç hiperparametre kombinasyonu deneyecek. Her deneme 5 katlı CV koşar; süreyi doğrusal artırır. Derste 20, kendi çalışmanızda 100–300 tipiktir. |
| `Y_RANDOM_TEKRAR` | y-randomizasyon tekrar sayısı. Literatürde 10 yaygın. |
| `ECFP_TARAMASI` | ECFP4 (2048 özellik) taraması belirgin biçimde yavaştır; ders akışını bekletmemek için kapalı. |

> ⏱️ **Süre sıkışırsa** `OPTUNA_DENEME = 10` ve `Y_RANDOM_TEKRAR = 5` yapın.
> Süre yarıdan aza iner, kavramlar aynı kalır; yalnızca bulunan
> hiperparametreler bir miktar daha zayıf olur.

In [ ]:
# ── Hedef ─────────────────────────────────────────────────────────────────────
TARGET_ADI  = "NTRK1"
HEDEF_KOLON = "pIC50"

# ── Google Drive proje klasoru (Oturum 1-2 ile AYNI) ──────────────────────────
DRIVE_PROJE_KLASORU = "/content/drive/MyDrive/EBYUSEM2026_MLQSAR"
YEREL_KLASOR        = "EBYUSEM2026_MLQSAR"
DEPO_RAW            = "https://raw.githubusercontent.com/eygpcr/ebyusem2026-mlqsar/main"

# ── Girdi dosyalari (Oturum 2 ciktilari) ─────────────────────────────────────
AD_DESC  = "ntrk1_descriptors.csv"
AD_ECFP  = "ntrk1_ecfp4.csv"
AD_MACCS = "ntrk1_maccs.csv"

# ── Cikti dosyalari ──────────────────────────────────────────────────────────
AD_SONUC   = "ntrk1_regresyon_sonuclari.csv"
AD_MODEL   = "ntrk1_en_iyi_regresyon_modeli.joblib"
AD_BOLUNME = "ntrk1_egitim_test_bolunmesi.csv"

# ── Modelleme ayarlari ───────────────────────────────────────────────────────
OZELLIK_SETI    = "descriptors"   # "descriptors" | "ecfp4" | "maccs"
TEST_ORANI      = 0.20            # tutulan test seti
DOGRULAMA_ORANI = 0.25            # egitimden ayrilan dogrulama (tarama icin)
CV_KAT          = 5               # K-Fold kat sayisi
OPTUNA_DENEME   = 20              # Optuna deneme sayisi
Y_RANDOM_TEKRAR = 10              # y-randomizasyon tekrar sayisi
RASTGELE_TOHUM  = 42

ECFP_TARAMASI = False   # True yaparsaniz ECFP4 taramasi da kosar (yavas)

print(f"Hedef          : {TARGET_ADI} — {HEDEF_KOLON} (regresyon)")
print(f"Ozellik seti   : {OZELLIK_SETI}")
print(f"Test orani     : %{TEST_ORANI * 100:.0f}")
print(f"Capraz dogrulama: {CV_KAT} kat")
print(f"Optuna denemesi: {OPTUNA_DENEME}")

---
## 2. Drive ve kurulum

Üç yeni kütüphane:

- **`lazypredict`** — tek komutla onlarca scikit-learn modelini eğitip
  karşılaştırır. Model seçiminde ilk eleme için idealdir.
- **`optuna`** — hiperparametre optimizasyonu. Izgara aramasından farkı: hangi
  bölgelerin umut verici olduğunu öğrenerek arar.
- **`xgboost`** — gradyan artırmalı ağaçlar; tablo verisinde en güçlü
  algoritmalardan biri.

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJE, ORTAM = Path(DRIVE_PROJE_KLASORU), "Colab + Drive"
except (ImportError, ModuleNotFoundError):
    PROJE, ORTAM = Path(YEREL_KLASOR), "Yerel"

KLASOR = {
    1: PROJE / "01_veri",
    2: PROJE / "02_ozellikler",
    3: PROJE / "03_regresyon",
    4: PROJE / "04_siniflandirma",
    5: PROJE / "05_analiz",
    6: PROJE / "06_adaylar",
}
for k in KLASOR.values():
    (k / "gorseller").mkdir(parents=True, exist_ok=True)

BU     = KLASOR[3]             # bu defterin klasoru
GORSEL = BU / "gorseller"

print(f"Ortam       : {ORTAM}")
print(f"Ana klasor  : {PROJE}")
print(f"Bu oturum   : {BU}")
print(f"Oturum 2'den okunacak: {KLASOR[2]}")

In [ ]:
!pip install lazypredict optuna xgboost -q
print("Kurulum tamam.")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import time

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (KFold, cross_val_predict, cross_val_score,
                                     train_test_split)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor,
                              HistGradientBoostingRegressor)
from xgboost import XGBRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from rdkit import Chem, RDLogger
from rdkit.Chem.Scaffolds import MurckoScaffold
RDLogger.DisableLog("rdApp.*")

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 25)

print(f"optuna {optuna.__version__} hazir.")

---
## 3. Özellik setlerini yükleme

Oturum 2'nin ürettiği üç dosyayı okuyoruz. Her dosyanın düzeni aynıdır:
`Smiles` solda, özellikler ortada, `pIC50` ve `Aktiflik` sağda.

In [ ]:
def ozellik_seti_yukle(dosya_adi: str):
    """Ozellik setini Drive'dan (yoksa depodan) okur; (X, y, smiles) dondurur."""
    yol = KLASOR[2] / dosya_adi
    if yol.exists():
        tablo = pd.read_csv(yol)
        kaynak = f"Drive: {yol.name}"
    else:
        raise FileNotFoundError(
            f"'{dosya_adi}' bulunamadi.\n"
            f"Once Oturum 2 (02_ozellik_uretimi.ipynb) not defterini ayni "
            f"Drive klasorunu kullanarak calistirin.\n"
            f"Beklenen yol: {yol}"
        )
    ozellikler = [k for k in tablo.columns
                  if k not in ("Smiles", HEDEF_KOLON, "Aktiflik")]
    return (tablo[ozellikler], tablo[HEDEF_KOLON], tablo["Smiles"], kaynak)


setler = {}
for ad, dosya in [("descriptors", AD_DESC), ("ecfp4", AD_ECFP), ("maccs", AD_MACCS)]:
    X, y, smi, kaynak = ozellik_seti_yukle(dosya)
    setler[ad] = {"X": X, "y": y, "smiles": smi}
    print(f"  {ad:<14} {X.shape[0]:>5} molekul x {X.shape[1]:>5} ozellik   ({kaynak})")

# Hepsinde ayni molekuller ve ayni hedef olmali
assert all(len(s["y"]) == len(setler["descriptors"]["y"]) for s in setler.values())
y_hedef = setler["descriptors"]["y"]
smiles  = setler["descriptors"]["smiles"]

print(f"\n{HEDEF_KOLON}: ortalama {y_hedef.mean():.2f}, "
      f"std {y_hedef.std():.2f}, aralik {y_hedef.min():.2f}-{y_hedef.max():.2f}")

---
## 4. Eğitim/test bölünmesi: rastgele mi, iskelet bazlı mı?

Bu, oturumun en önemli kavramsal noktası.

### Rastgele bölünmenin sorunu

Bir ilaç keşif programı tek bir **kimyasal seriyi** optimize eder: aynı iskelet
etrafında yüzlerce türev sentezlenir. Oturum 1'de en potent moleküllere
baktığımızda bunu görmüştük.

Rastgele böldüğünüzde aynı serinin molekülleri hem eğitime hem teste dağılır.
Model test setindeki bir molekülü tahmin ederken eğitimde onun **neredeyse
birebir aynısını** görmüştür. Bu ezberlemeyi ödüllendirir ve test skorunu
şişirir.

Oysa modelden beklediğimiz şey farklıdır: **daha önce görmediği bir kimyasal
seriye** ait molekül hakkında isabetli tahmin yapmak.

### İskelet bazlı bölünme

Çözüm: molekülleri **Murcko iskeletlerine** göre gruplayıp bir iskeletin bütün
moleküllerini ya eğitime ya teste vermek. Böylece test seti model için gerçekten
yeni yapılar içerir.

**Beklenti:** iskelet bazlı bölünmede performans **daha düşük** çıkar. Bu bir
başarısızlık değil, daha dürüst bir ölçümdür. Literatürdeki yüksek R²
değerlerinin bir kısmı bu etkiden kaynaklanır.

Aşağıda ikisini de kurup yan yana karşılaştıracağız.

In [ ]:
def halka_cercevesi(mol) -> str:
    """Yan zincirleri kirparak halka cercevesini bulur (yedek yol).

    Halkada olmayan uc atomlar, hic kalmayana kadar tek tek silinir. Geriye
    halkalar ve onlari birbirine baglayan koprüler kalir -- Murcko iskeletinin
    tanimi da budur.
    """
    duzen = Chem.RWMol(mol)
    while True:
        atilacak = [a.GetIdx() for a in duzen.GetAtoms()
                    if a.GetDegree() == 1 and not a.IsInRing()]
        if not atilacak:
            break
        for indeks in sorted(atilacak, reverse=True):
            duzen.RemoveAtom(indeks)
    cerceve = duzen.GetMol()
    try:
        Chem.SanitizeMol(cerceve)
    except Exception:
        pass
    try:
        return Chem.MolToSmiles(cerceve)
    except Exception:
        return ""


# RDKit'in bazi surumlerinde MurckoScaffoldSmiles hata veriyor. Once onu
# deniyor, calismazsa yukaridaki yedek yola gecip bir kez haber veriyoruz.
_YEDEK_ISKELET = {"kullanildi": False}


def murcko_iskeleti(smi: str) -> str:
    """Molekulun Murcko iskeletini (halka cercevesini) dondurur."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return ""
    if not _YEDEK_ISKELET["kullanildi"]:
        try:
            return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
        except Exception:
            _YEDEK_ISKELET["kullanildi"] = True
            print("Not: RDKit'in MurckoScaffold islevi bu surumde hata verdi;")
            print("     iskeletler yedek yolla (yan zincir kirpma) bulunuyor.")
    return halka_cercevesi(mol)


def iskelet_bolunmesi(smiles_dizisi, test_orani=0.2, tohum=42):
    """Iskelet bazli egitim/test bolunmesi.

    Her iskelet grubunun TAMAMI tek bir tarafa gider. Gruplar buyukten kucuge
    siralanir; en kalabalik iskeletler egitime, kalanlar hedef test oranina
    ulasilana kadar teste atanir.
    """
    iskeletler = [murcko_iskeleti(s) for s in smiles_dizisi]
    gruplar = {}
    for indeks, iskelet in enumerate(iskeletler):
        gruplar.setdefault(iskelet, []).append(indeks)

    # Buyukten kucuge sirala (ayni boyuttakiler icin tekrarlanabilir sira)
    sirali = sorted(gruplar.values(), key=lambda g: (-len(g), g[0]))

    hedef_test = int(len(smiles_dizisi) * test_orani)
    egitim_idx, test_idx = [], []
    for grup in sirali:
        if len(test_idx) + len(grup) <= hedef_test:
            test_idx.extend(grup)
        else:
            egitim_idx.extend(grup)
    return np.array(sorted(egitim_idx)), np.array(sorted(test_idx)), iskeletler


# --- Rastgele bolunme ---
idx_hepsi = np.arange(len(y_hedef))
r_egitim, r_test = train_test_split(
    idx_hepsi, test_size=TEST_ORANI, random_state=RASTGELE_TOHUM
)

# --- Iskelet bazli bolunme ---
i_egitim, i_test, iskeletler = iskelet_bolunmesi(
    smiles, test_orani=TEST_ORANI, tohum=RASTGELE_TOHUM
)

print("Bolunme karsilastirmasi")
print("-" * 68)
print(f"{'':22}{'Egitim':>10}{'Test':>10}{'Test iskeleti':>16}{'Ortak iskelet':>16}")
for ad, (eg, te) in [("Rastgele", (r_egitim, r_test)),
                     ("Iskelet bazli", (i_egitim, i_test))]:
    isk_eg = {iskeletler[i] for i in eg}
    isk_te = {iskeletler[i] for i in te}
    print(f"{ad:<22}{len(eg):>10}{len(te):>10}{len(isk_te):>16}"
          f"{len(isk_eg & isk_te):>16}")

print(f"\nToplam benzersiz iskelet: {len(set(iskeletler))}")
print("\nDikkat: rastgele bolunmede test iskeletlerinin neredeyse tamami")
print("egitimde de bulunur. Iskelet bazli bolunmede ORTAK ISKELET SIFIRDIR:")
print("test seti modelin hic gormedigi yapilardan olusur.")

### Eğitim içinden doğrulama seti ayırmak

Model **taraması** için bir değerlendirme setine ihtiyacımız var — ama test
setini kullanamayız.

Bu yüzden eğitim verisini ikiye ayırıyoruz: **iç-eğitim** ve **doğrulama**.
Tarama doğrulama seti üzerinde yapılacak; test seti dokunulmadan bekleyecek.

> **LazyPredict hakkında uyarı.** `LazyRegressor.fit()` imzası
> `fit(X_train, X_test, y_train, y_test)` biçimindedir ve verdiğiniz "test"
> setinde skor üretir. Oraya gerçek test setinizi verip sonra en iyi modeli o
> skora göre seçerseniz, bildirdiğiniz test performansı **iyimser yanlı** olur.
> Bu yüzden biz oraya **doğrulama** setini veriyoruz.

In [ ]:
# Varsayilan olarak RASTGELE bolunmeyle ilerliyoruz; iskelet bazli bolunmeyi
# Bolum 9'da ayni modelle tekrar kurup karsilastiracagiz.
egitim_idx, test_idx = r_egitim, r_test

# Egitim -> ic-egitim + dogrulama
ic_egitim_idx, dogrulama_idx = train_test_split(
    egitim_idx, test_size=DOGRULAMA_ORANI, random_state=RASTGELE_TOHUM
)

print(f"Toplam molekul : {len(y_hedef)}")
print(f"  Egitim       : {len(egitim_idx):>5}  (%{100 * len(egitim_idx) / len(y_hedef):.0f})")
print(f"    ic-egitim  : {len(ic_egitim_idx):>5}")
print(f"    dogrulama  : {len(dogrulama_idx):>5}")
print(f"  TEST (tutulu): {len(test_idx):>5}  (%{100 * len(test_idx) / len(y_hedef):.0f})")

# Bolunmeyi kaydet: sonraki oturumlar AYNI bolunmeyi kullanacak
bolunme_tablosu = pd.DataFrame({
    "Smiles": smiles,
    HEDEF_KOLON: y_hedef,
    "iskelet": iskeletler,
    "rastgele_bolum": ["test" if i in set(test_idx) else "egitim"
                       for i in idx_hepsi],
    "iskelet_bolum": ["test" if i in set(i_test) else "egitim"
                      for i in idx_hepsi],
})
bolunme_tablosu.to_csv(BU / AD_BOLUNME, index=False)
print(f"\nBolunme kaydedildi: {BU / AD_BOLUNME}")

---
## 5. LazyPredict ile hızlı model taraması

Hangi algoritma ailesiyle çalışacağımıza karar vermemiz gerekiyor: lineer
modeller, ağaçlar, destek vektör makineleri, komşuluk tabanlı yöntemler...

Hepsini elle denemek yerine **LazyPredict** kullanıyoruz: tek komutla ~40
regresörü varsayılan ayarlarıyla eğitip bir tabloda sıralar.

**Ne için iyi?** Hızlı **eleme**. Hangi ailelerin bu veride umut verdiğini
dakikalar içinde gösterir.

**Ne için iyi değil?** Son karar. Çünkü modeller **varsayılan**
hiperparametrelerle çalışır (ayarlanmış bir model sıralamayı değiştirebilir) ve
ölçüm tek bir bölünme üzerinde yapılır (çapraz doğrulama yok, sıralama
gürültülü).

Akış şöyle: **LazyPredict ile ele → Optuna ile ayarla → çapraz doğrulama ile
doğrula → test setinde bir kez ölç.**

In [ ]:
from lazypredict.Supervised import LazyRegressor


def tarama_yap(X, y, ic_idx, dog_idx, etiket):
    """LazyPredict ile hizli model taramasi; dogrulama setinde skorlar."""
    X_ic,  y_ic  = X.iloc[ic_idx],  y.iloc[ic_idx]
    X_dog, y_dog = X.iloc[dog_idx], y.iloc[dog_idx]

    baslangic = time.time()
    tarayici = LazyRegressor(verbose=0, ignore_warnings=True, predictions=False)
    tablo, _ = tarayici.fit(X_ic, X_dog, y_ic, y_dog)
    sure = time.time() - baslangic

    tablo = tablo.copy()
    tablo["Ozellik seti"] = etiket
    print(f"  {etiket:<14} {len(tablo):>3} model, {sure:>6.1f} s   "
          f"en iyi: {tablo.index[0]} (R2={tablo['R-Squared'].iloc[0]:.3f})")
    return tablo


print("LazyPredict taramasi basliyor (dogrulama seti uzerinde)...")
print("-" * 76)

taramalar = []
taramalar.append(tarama_yap(setler["descriptors"]["X"], y_hedef,
                            ic_egitim_idx, dogrulama_idx, "descriptors"))
taramalar.append(tarama_yap(setler["maccs"]["X"], y_hedef,
                            ic_egitim_idx, dogrulama_idx, "maccs"))
if ECFP_TARAMASI:
    taramalar.append(tarama_yap(setler["ecfp4"]["X"], y_hedef,
                                ic_egitim_idx, dogrulama_idx, "ecfp4"))
else:
    print("  ecfp4          atlandi (ECFP_TARAMASI = False)")

tarama_hepsi = pd.concat(taramalar)
print("\nTarama tamamlandi.")

In [ ]:
# Secilen ozellik setindeki en iyi 12 model
tarama = (tarama_hepsi[tarama_hepsi["Ozellik seti"] == OZELLIK_SETI]
          .sort_values("R-Squared", ascending=False))

print(f"LazyPredict siralamasi — '{OZELLIK_SETI}' (dogrulama seti)")
print("=" * 72)
print(tarama[["R-Squared", "RMSE", "Time Taken"]].head(12).round(4).to_string())

print(f"\nEn iyi model: {tarama.index[0]}")

### Bütün modellerin sonuçları — bar grafikleri

Tablo ilk 12 modeli gösterdi. Aşağıda **taranan bütün algoritmaların**
sonuçları iki grafikte:

- **Solda R²** — yüksek olan iyi. 1'e yakın = hedefteki değişkenliğin büyük
  bölümünü açıklıyor.
- **Sağda RMSE** — düşük olan iyi. `pIC50` biriminde ortalama hata.

Yeşil çubuk, bir sonraki bölümde Optuna ile ayarlayacağımız modeli gösteriyor.

Grafikteki en öğretici şey **dağılımın kendisi**: üst grupta birbirine çok
yakın onlarca model var. Yani "en iyi model" seçimi, çoğu zaman birkaç binde
birlik farklara dayanır ve başka bir bölünmede sıra değişebilir. Bu yüzden
LazyPredict bir **eleme** aracıdır, karar verici değil.

In [ ]:
# Cok kotu modeller ekseni ezmesin diye bir alt sinir koyuyoruz.
ALT_SINIR_R2 = -0.5

tum_modeller = tarama.sort_values("R-Squared")
kirpilan = tum_modeller[tum_modeller["R-Squared"] < ALT_SINIR_R2]
cizilecek = tum_modeller[tum_modeller["R-Squared"] >= ALT_SINIR_R2]

en_iyi_ad = tarama.index[0]
renkler = ["#27ae60" if ad == en_iyi_ad else "#7f8c8d"
           for ad in cizilecek.index]

fig, (sol, sag) = plt.subplots(1, 2, figsize=(14, 0.27 * len(cizilecek) + 2.2),
                               sharey=True)
y = np.arange(len(cizilecek))

sol.barh(y, cizilecek["R-Squared"], color=renkler, edgecolor="white", height=0.72)
for i, v in enumerate(cizilecek["R-Squared"]):
    sol.text(v + 0.012, i, f"{v:.3f}", va="center", fontsize=7.5, color="#444")
sol.set_yticks(y, cizilecek.index, fontsize=7.5)
sol.set_xlabel("R2  (yuksek = iyi)")
sol.set_title(f"R2 — {OZELLIK_SETI} (dogrulama seti)")
sol.set_xlim(min(0, cizilecek["R-Squared"].min()) - 0.05,
             cizilecek["R-Squared"].max() + 0.12)
sol.axvline(0, color="k", linewidth=0.8)
sol.grid(alpha=0.3, axis="x", linestyle="--")

sag.barh(y, cizilecek["RMSE"], color=renkler, edgecolor="white", height=0.72)
for i, v in enumerate(cizilecek["RMSE"]):
    sag.text(v + cizilecek["RMSE"].max() * 0.012, i, f"{v:.3f}",
             va="center", fontsize=7.5, color="#444")
sag.set_xlabel(f"RMSE  ({HEDEF_KOLON} birimi, dusuk = iyi)")
sag.set_title(f"RMSE — {OZELLIK_SETI} (dogrulama seti)")
sag.set_xlim(0, cizilecek["RMSE"].max() * 1.16)
sag.grid(alpha=0.3, axis="x", linestyle="--")

fig.suptitle(f"LazyPredict — taranan {len(tarama)} modelin tamami",
             fontsize=12, y=1.004)
plt.tight_layout()
plt.savefig(GORSEL / "03_lazypredict_butun_modeller.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"Cizilen model   : {len(cizilecek)} / {len(tarama)}")
if len(kirpilan):
    print(f"Grafik disi     : {len(kirpilan)} model (R2 < {ALT_SINIR_R2})")
    print("  " + ", ".join(f"{ad} ({r:.1f})" for ad, r
                           in kirpilan["R-Squared"].items()))
print(f"\nEn iyi 5 modelin R2 araligi: "
      f"{tarama['R-Squared'].head(5).min():.4f} - "
      f"{tarama['R-Squared'].head(5).max():.4f}")
print("Ustteki modeller birbirine cok yakin: tek bir bolunmedeki kucuk")
print("farklar siralamayi degistirebilir.")

In [ ]:
# Ozellik setleri yan yana: hangi TEMSIL daha iyi?
fig, eksen = plt.subplots(figsize=(11, 4.6))

for etiket, grup in tarama_hepsi.groupby("Ozellik seti"):
    ilk10 = grup.sort_values("R-Squared", ascending=False).head(10)
    eksen.plot(range(1, len(ilk10) + 1), ilk10["R-Squared"],
               marker="o", label=f"{etiket} (en iyi R2={ilk10['R-Squared'].iloc[0]:.3f})")

eksen.set_xlabel("Siralama (en iyi -> 10.)")
eksen.set_ylabel("R2 (dogrulama seti)")
eksen.set_title("Ozellik setlerinin karsilastirmasi — LazyPredict ilk 10 model")
eksen.legend()
eksen.grid(alpha=0.3, linestyle="--")
plt.tight_layout()
plt.show()

ozet_tarama = (tarama_hepsi.groupby("Ozellik seti")["R-Squared"]
               .agg(["max", "median", "count"])
               .rename(columns={"max": "En iyi R2", "median": "Medyan R2",
                                "count": "Model sayisi"})
               .sort_values("En iyi R2", ascending=False))
print(ozet_tarama.round(4).to_string())

---
## 6. Optuna ile hiperparametre optimizasyonu

LazyPredict umut verici bir model ailesi gösterdi. Şimdi o ailenin
**hiperparametrelerini** ayarlayacağız.

**Hiperparametre nedir?** Modelin veriden *öğrenmediği*, bizim belirlediğimiz
ayarlar: ağaç sayısı, ağaç derinliği, öğrenme hızı, düzenlileştirme gücü.

**Neden Optuna?** Izgara araması bütün kombinasyonları körlemesine dener;
parametre sayısı arttıkça maliyeti patlar. Optuna hangi bölgelerin iyi sonuç
verdiğini öğrenir ve aramayı oraya yoğunlaştırır.

**Amaç fonksiyonu:** eğitim verisi üzerinde **5 katlı çapraz doğrulama R²**'si.
Tek bir doğrulama skoruna göre ayar yapmak gürültülüdür; çapraz doğrulama beş
bölünmenin ortalamasını alır.

> **Ölçekleme nerede?** Her modeli bir `Pipeline` içine koyuyoruz:
> `StandardScaler` → model. Böylece çapraz doğrulamanın **her katmanında**
> ölçekleyici yalnızca o katmanın eğitim kısmına uydurulur. Oturum 2'de
> anlattığımız sızıntı sorunu böyle çözülür.

In [ ]:
# Her model ailesi icin arama uzayi.
# Donen deger: (pipeline, parametre sozlugu)
def _pipeline(model, olcekle: bool):
    adimlar = ([("olcekle", StandardScaler())] if olcekle else []) + [("model", model)]
    return Pipeline(adimlar)


def uzay_random_forest(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 4, 30),
        min_samples_split=deneme.suggest_int("min_samples_split", 2, 12),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 1, 8),
        max_features=deneme.suggest_float("max_features", 0.1, 1.0),
    )
    return _pipeline(RandomForestRegressor(random_state=RASTGELE_TOHUM, n_jobs=-1, **p),
                 olcekle=False), p


def uzay_extra_trees(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 4, 30),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 1, 8),
        max_features=deneme.suggest_float("max_features", 0.1, 1.0),
    )
    return _pipeline(ExtraTreesRegressor(random_state=RASTGELE_TOHUM, n_jobs=-1, **p),
                 olcekle=False), p


def uzay_xgboost(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 800, step=50),
        max_depth=deneme.suggest_int("max_depth", 2, 12),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=deneme.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=deneme.suggest_float("colsample_bytree", 0.3, 1.0),
        reg_lambda=deneme.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        min_child_weight=deneme.suggest_int("min_child_weight", 1, 10),
    )
    return _pipeline(XGBRegressor(random_state=RASTGELE_TOHUM, n_jobs=-1,
                              tree_method="hist", **p), olcekle=False), p


def uzay_hist_gb(deneme):
    p = dict(
        max_iter=deneme.suggest_int("max_iter", 100, 600, step=50),
        max_depth=deneme.suggest_int("max_depth", 3, 20),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        min_samples_leaf=deneme.suggest_int("min_samples_leaf", 5, 40),
        l2_regularization=deneme.suggest_float("l2_regularization", 1e-3, 10.0, log=True),
    )
    return _pipeline(HistGradientBoostingRegressor(random_state=RASTGELE_TOHUM, **p),
                 olcekle=False), p


def uzay_svr(deneme):
    p = dict(
        C=deneme.suggest_float("C", 0.1, 300, log=True),
        gamma=deneme.suggest_float("gamma", 1e-4, 1.0, log=True),
        epsilon=deneme.suggest_float("epsilon", 0.01, 1.0),
    )
    return _pipeline(SVR(kernel="rbf", **p), olcekle=True), p


def uzay_knn(deneme):
    p = dict(
        n_neighbors=deneme.suggest_int("n_neighbors", 1, 30),
        weights=deneme.suggest_categorical("weights", ["uniform", "distance"]),
        p=deneme.suggest_int("p", 1, 2),
    )
    return _pipeline(KNeighborsRegressor(n_jobs=-1, **p), olcekle=True), p


def uzay_ridge(deneme):
    p = dict(alpha=deneme.suggest_float("alpha", 1e-3, 1000, log=True))
    return _pipeline(Ridge(random_state=RASTGELE_TOHUM, **p), olcekle=True), p


def uzay_gradient_boosting(deneme):
    p = dict(
        n_estimators=deneme.suggest_int("n_estimators", 100, 500, step=50),
        max_depth=deneme.suggest_int("max_depth", 2, 10),
        learning_rate=deneme.suggest_float("learning_rate", 0.01, 0.3, log=True),
        subsample=deneme.suggest_float("subsample", 0.5, 1.0),
    )
    return _pipeline(GradientBoostingRegressor(random_state=RASTGELE_TOHUM, **p),
                 olcekle=False), p


# LazyPredict model adi -> arama uzayi
ARAMA_UZAYLARI = {
    "RandomForestRegressor":           uzay_random_forest,
    "ExtraTreesRegressor":             uzay_extra_trees,
    "XGBRegressor":                    uzay_xgboost,
    "LGBMRegressor":                   uzay_hist_gb,   # yoksa HistGB'ye esle
    "HistGradientBoostingRegressor":   uzay_hist_gb,
    "SVR":                             uzay_svr,
    "NuSVR":                           uzay_svr,
    "KNeighborsRegressor":             uzay_knn,
    "Ridge":                           uzay_ridge,
    "RidgeCV":                         uzay_ridge,
    "BaggingRegressor":                uzay_random_forest,
    "GradientBoostingRegressor":       uzay_gradient_boosting,
}

print(f"Tanimli arama uzayi: {len(ARAMA_UZAYLARI)} model ailesi")

In [ ]:
# LazyPredict siralamasindan, arama uzayi TANIMLI olan en iyi modeli sec
secilen_ad, secilen_uzay = None, None
for model_adi in tarama.index:
    if model_adi in ARAMA_UZAYLARI:
        secilen_ad, secilen_uzay = model_adi, ARAMA_UZAYLARI[model_adi]
        break

if secilen_ad is None:
    secilen_ad, secilen_uzay = "XGBRegressor", uzay_xgboost
    print("Taramada arama uzayi tanimli model bulunamadi; XGBoost'a donuluyor.")

sira = list(tarama.index).index(secilen_ad) + 1
print(f"Optimize edilecek model : {secilen_ad}")
print(f"LazyPredict siralamasi  : {sira}.")
print(f"Tarama R2 (varsayilan)  : {tarama.loc[secilen_ad, 'R-Squared']:.4f}")

# Modelleme verisi
X_tam = setler[OZELLIK_SETI]["X"]
X_egitim, y_egitim = X_tam.iloc[egitim_idx], y_hedef.iloc[egitim_idx]
X_test,   y_test   = X_tam.iloc[test_idx],  y_hedef.iloc[test_idx]

print(f"\nEgitim: {X_egitim.shape}   Test: {X_test.shape}")

In [ ]:
kfold = KFold(n_splits=CV_KAT, shuffle=True, random_state=RASTGELE_TOHUM)


def amac(deneme):
    """Optuna amac fonksiyonu: egitim verisinde 5 katli CV R2 (buyuk = iyi)."""
    pipeline, _ = secilen_uzay(deneme)
    skorlar = cross_val_score(pipeline, X_egitim, y_egitim, cv=kfold,
                              scoring="r2", n_jobs=1)
    return skorlar.mean()


print(f"Optuna optimizasyonu: {OPTUNA_DENEME} deneme x {CV_KAT} katli CV")
print("(birkac dakika surebilir)\n")

baslangic = time.time()
calisma = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RASTGELE_TOHUM),
    study_name=f"{secilen_ad}_regresyon",
)
calisma.optimize(amac, n_trials=OPTUNA_DENEME, show_progress_bar=False)
sure = time.time() - baslangic

print(f"Tamamlandi: {sure:.1f} s  ({sure / OPTUNA_DENEME:.1f} s/deneme)")
print(f"\nEn iyi CV R2 : {calisma.best_value:.4f}")
print("En iyi parametreler:")
for anahtar, deger in calisma.best_params.items():
    print(f"  {anahtar:<22} {deger}")

In [ ]:
# Optimizasyon gecmisi: Optuna gercekten iyilestirdi mi?
degerler = [d.value for d in calisma.trials if d.value is not None]
en_iyi_kumulatif = np.maximum.accumulate(degerler)

fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.2))

sol.plot(range(1, len(degerler) + 1), degerler, "o", alpha=0.5,
         label="deneme skoru", color="#7f8c8d")
sol.plot(range(1, len(degerler) + 1), en_iyi_kumulatif, "-", linewidth=2,
         label="o ana kadarki en iyi", color="#c0392b")
sol.set_xlabel("Deneme")
sol.set_ylabel(f"{CV_KAT} katli CV R2")
sol.set_title(f"Optuna optimizasyon gecmisi — {secilen_ad}")
sol.legend()
sol.grid(alpha=0.3, linestyle="--")

# Parametre onemi (yeterli deneme varsa)
try:
    onem = optuna.importance.get_param_importances(calisma)
    adlar = list(onem.keys())[::-1]
    paylar = [onem[a] for a in adlar]
    sag.barh(adlar, paylar, color="#4878a8")
    sag.set_xlabel("Onem (fANOVA)")
    sag.set_title("Hangi hiperparametre onemliydi?")
    sag.grid(alpha=0.3, axis="x", linestyle="--")
except Exception as hata:
    sag.text(0.5, 0.5, f"Onem hesaplanamadi:\n{hata}",
             ha="center", va="center", transform=sag.transAxes)
    sag.axis("off")

plt.tight_layout()
plt.show()

---
## 7. Son modeli eğitmek ve test setinde **bir kez** ölçmek

En iyi hiperparametreleri biliyoruz. Son adımlar:

1. Modeli **bütün eğitim verisiyle** yeniden eğit.
2. **Test setinde** performansı ölç — test setine ilk ve son dokunuşumuz.

### Raporladığımız metrikler

| Metrik | Ne söyler? |
|---|---|
| **R²** | Hedefteki değişkenliğin ne kadarını açıklıyor? (1 = kusursuz, 0 = ortalama tahmin kadar) |
| **Q²** (CV R²) | Çapraz doğrulama R²'si — modelin *görmediği* veride beklenen performans |
| **RMSE** | Ortalama hata, `pIC50` biriminde. Büyük hataları çok cezalandırır |
| **MAE** | Ortalama mutlak hata — yorumlaması en kolay |

**RMSE'yi nasıl okumalı?** `pIC50` logaritmik bir ölçek: 1 birim hata,
aktivitede **10 kat** yanılma demektir. RMSE ≈ 0,5 ise tipik tahminler ~3 kat
içinde isabet ediyor; ilaç keşfinde bu kullanışlı bir doğruluktur.
Karşılaştırma noktası olarak hedefin standart sapmasına bakın: RMSE o sayıdan
belirgin biçimde küçük olmalı.

In [ ]:
# En iyi parametrelerle pipeline'i yeniden kur
en_iyi_pipeline, _ = secilen_uzay(optuna.trial.FixedTrial(calisma.best_params))

# 1) Capraz dogrulama tahminleri (egitim verisi uzerinde, sizintisiz)
cv_tahmin = cross_val_predict(en_iyi_pipeline, X_egitim, y_egitim, cv=kfold, n_jobs=1)

# 2) Butun egitim verisiyle son egitim
en_iyi_pipeline.fit(X_egitim, y_egitim)

# 3) Tahminler
tahmin_egitim = en_iyi_pipeline.predict(X_egitim)
tahmin_test   = en_iyi_pipeline.predict(X_test)      # test setine ILK ve TEK dokunus


def metrikler(gercek, tahmin) -> dict:
    return {
        "R2":   r2_score(gercek, tahmin),
        "RMSE": float(np.sqrt(mean_squared_error(gercek, tahmin))),
        "MAE":  mean_absolute_error(gercek, tahmin),
    }


m_egitim = metrikler(y_egitim, tahmin_egitim)
m_cv     = metrikler(y_egitim, cv_tahmin)
m_test   = metrikler(y_test, tahmin_test)

sonuc = pd.DataFrame(
    [m_egitim, m_cv, m_test],
    index=["Egitim (yeniden uyum)", f"Capraz dogrulama (Q2, {CV_KAT} kat)",
           "TEST (tutulan)"],
)

print(f"Model: {secilen_ad}   |   Ozellik seti: {OZELLIK_SETI}")
print("=" * 66)
print(sonuc.round(4).to_string())

print(f"\nKarsilastirma noktasi: {HEDEF_KOLON} standart sapmasi = "
      f"{y_hedef.std():.3f}")
print(f"Test RMSE / std       = {m_test['RMSE'] / y_hedef.std():.3f}"
      "   (1'den ne kadar kucukse o kadar iyi)")

fark = m_egitim["R2"] - m_cv["R2"]
print(f"\nEgitim R2 - CV R2 = {fark:.3f}")
if fark > 0.25:
    print("  -> Belirgin asiri ogrenme (overfitting) isareti: model egitim")
    print("     verisini ezberliyor. Duzenlilestirmeyi artirmayi dusunun.")
else:
    print("  -> Egitim ve CV performansi makul olcude yakin.")

In [ ]:
fig, eksenler = plt.subplots(1, 3, figsize=(15.5, 4.8))

alt = min(y_hedef.min(), tahmin_test.min()) - 0.3
ust = max(y_hedef.max(), tahmin_test.max()) + 0.3

for eksen, (gercek, tahmin, baslik, m, renk) in zip(eksenler, [
    (y_egitim, tahmin_egitim, "Egitim (yeniden uyum)", m_egitim, "#4878a8"),
    (y_egitim, cv_tahmin,     f"Capraz dogrulama (Q2)", m_cv,     "#d68910"),
    (y_test,   tahmin_test,   "TEST (tutulan)",         m_test,   "#27ae60"),
]):
    eksen.scatter(gercek, tahmin, s=9, alpha=0.4, color=renk)
    eksen.plot([alt, ust], [alt, ust], "k--", linewidth=1, label="kusursuz tahmin")
    eksen.set_xlim(alt, ust)
    eksen.set_ylim(alt, ust)
    eksen.set_xlabel(f"Deneysel {HEDEF_KOLON}")
    eksen.set_ylabel(f"Tahmin edilen {HEDEF_KOLON}")
    eksen.set_title(f"{baslik}\nR2={m['R2']:.3f}  RMSE={m['RMSE']:.3f}  "
                    f"MAE={m['MAE']:.3f}", fontsize=10)
    eksen.legend(fontsize=8)
    eksen.grid(alpha=0.3, linestyle="--")

plt.suptitle(f"{secilen_ad} — {TARGET_ADI} {HEDEF_KOLON} tahmini", y=1.03)
plt.tight_layout()
plt.savefig(GORSEL / "01_tahmin_grafikleri.png", dpi=150,
            bbox_inches="tight")
plt.show()

In [ ]:
# Artik (residual) analizi: hatalar nerede yogunlasiyor?
artiklar = y_test.values - tahmin_test

fig, (sol, sag) = plt.subplots(1, 2, figsize=(13, 4.2))

sol.scatter(tahmin_test, artiklar, s=10, alpha=0.45, color="#27ae60")
sol.axhline(0, color="k", linestyle="--", linewidth=1)
sol.axhline(artiklar.std(), color="#c0392b", linestyle=":", linewidth=1,
            label=f"+/- 1 std = {artiklar.std():.2f}")
sol.axhline(-artiklar.std(), color="#c0392b", linestyle=":", linewidth=1)
sol.set_xlabel(f"Tahmin edilen {HEDEF_KOLON}")
sol.set_ylabel("Artik (gercek - tahmin)")
sol.set_title("Artiklar tahmin degerine gore")
sol.legend(fontsize=8)
sol.grid(alpha=0.3, linestyle="--")

sag.hist(artiklar, bins=35, color="#27ae60", edgecolor="white", alpha=0.85)
sag.axvline(0, color="k", linestyle="--", linewidth=1)
sag.set_xlabel("Artik")
sag.set_ylabel("Molekul sayisi")
sag.set_title(f"Artik dagilimi  (ortalama={artiklar.mean():.3f}, "
              f"std={artiklar.std():.3f})")
sag.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.show()

print("Ne aramaliyiz?")
print("  - Artiklar 0 cevresinde rastgele dagilmali (sistematik egilim olmamali).")
print("  - Sol grafikte asagi/yukari dogru bir 'huni' varsa, model potens")
print("    araliginin bir ucunda digerinden daha kotu tahmin yapiyor demektir.")
print(f"  - Buyuk hatali molekuller ({int((np.abs(artiklar) > 1).sum())} adet, "
      f"|artik| > 1) Oturum 5'te uygulanabilirlik alani ile incelenecek.")

---
## 8. y-randomizasyon: model gerçekten öğrendi mi?

QSAR'da **zorunlu** sayılan bir doğrulama testi. Mantığı çok basit:

> Hedef değişkeni (`pIC50`) moleküller arasında **rastgele karıştırın** ve
> modeli yeniden eğitin. Artık yapı ile aktivite arasında hiçbir gerçek ilişki
> yoktur. Model bu sahte veride de iyi skor alıyorsa, gerçek veride öğrendiği
> şey de kimya değil **gürültüymüş** demektir.

**Beklenen sonuç:** karıştırılmış hedeflerle R² ≈ 0 ya da negatif olmalı.
Gerçek model ile karıştırılmış modeller arasında büyük bir **uçurum** görmek
istiyoruz.

**Neden önemli?** Özellik sayısı molekül sayısına yaklaştığında (ör. 2048 bit,
500 molekül) modeller tesadüfi örüntüleri öğrenebilir. Bu teste "şans
korelasyonu kontrolü" de denir.

In [ ]:
print(f"y-randomizasyon: {Y_RANDOM_TEKRAR} tekrar")
print("(her tekrarda hedef karistirilir ve model yeniden egitilir)\n")

rastgele = np.random.RandomState(RASTGELE_TOHUM)
sahte_r2_cv, sahte_r2_test = [], []

baslangic = time.time()
for tekrar in range(Y_RANDOM_TEKRAR):
    y_karisik = pd.Series(
        rastgele.permutation(y_egitim.values), index=y_egitim.index
    )
    sahte_pipeline, _ = secilen_uzay(optuna.trial.FixedTrial(calisma.best_params))

    # Karistirilmis hedefle CV skoru
    skor = cross_val_score(sahte_pipeline, X_egitim, y_karisik, cv=kfold,
                           scoring="r2", n_jobs=1).mean()
    sahte_r2_cv.append(skor)

    # Karistirilmis hedefle egitip GERCEK test setinde dene
    sahte_pipeline.fit(X_egitim, y_karisik)
    sahte_r2_test.append(r2_score(y_test, sahte_pipeline.predict(X_test)))

    if (tekrar + 1) % 5 == 0:
        print(f"  {tekrar + 1:>2}/{Y_RANDOM_TEKRAR} tamamlandi "
              f"({time.time() - baslangic:.0f} s)")

sahte_r2_cv  = np.array(sahte_r2_cv)
sahte_r2_test = np.array(sahte_r2_test)

print(f"\n{'':26}{'CV R2':>12}{'Test R2':>12}")
print("-" * 50)
print(f"{'GERCEK model':<26}{m_cv['R2']:>12.4f}{m_test['R2']:>12.4f}")
print(f"{'Karistirilmis (ortalama)':<26}{sahte_r2_cv.mean():>12.4f}"
      f"{sahte_r2_test.mean():>12.4f}")
print(f"{'Karistirilmis (en iyi)':<26}{sahte_r2_cv.max():>12.4f}"
      f"{sahte_r2_test.max():>12.4f}")
print(f"{'Karistirilmis (std)':<26}{sahte_r2_cv.std():>12.4f}"
      f"{sahte_r2_test.std():>12.4f}")

ucurum = m_cv["R2"] - sahte_r2_cv.max()
print(f"\nGercek CV R2 ile en iyi karistirilmis CV R2 arasindaki fark: {ucurum:.4f}")
if ucurum > 0.2:
    print("  -> BASARILI. Model gercek bir yapi-aktivite iliskisi ogrenmis.")
else:
    print("  -> DIKKAT! Fark kucuk. Model sans korelasyonu ogrenmis olabilir.")

In [ ]:
fig, eksen = plt.subplots(figsize=(10, 4.4))

eksen.hist(sahte_r2_cv, bins=14, color="#95a5a6", edgecolor="white",
           alpha=0.85, label=f"Karistirilmis hedef (n={Y_RANDOM_TEKRAR})")
eksen.axvline(m_cv["R2"], color="#c0392b", linewidth=2.5,
              label=f"GERCEK model CV R2 = {m_cv['R2']:.3f}")
eksen.axvline(0, color="k", linestyle="--", linewidth=1, label="R2 = 0")
eksen.set_xlabel(f"{CV_KAT} katli capraz dogrulama R2")
eksen.set_ylabel("Tekrar sayisi")
eksen.set_title("y-randomizasyon — gercek model ile sans duzeyi arasindaki ucurum")
eksen.legend()
eksen.grid(alpha=0.3, axis="y", linestyle="--")

plt.tight_layout()
plt.savefig(GORSEL / "02_y_randomizasyon.png", dpi=150, bbox_inches="tight")
plt.show()

print("Gri dagilim 'sans duzeyini' gosterir. Kirmizi cizgi ondan ne kadar")
print("uzaksa, modelin ogrendigi iliski o kadar gercektir.")

---
## 9. Rastgele bölünme ile iskelet bazlı bölünmeyi karşılaştırmak

Şimdi Bölüm 4'te söz verdiğimiz karşılaştırmayı yapıyoruz: **aynı model, aynı
hiperparametreler**, yalnızca bölünme biçimi farklı.

**Beklenti:** iskelet bazlı bölünmede test performansı daha düşük olacak.
Aradaki fark, rastgele bölünmenin ne kadar **iyimser** olduğunu ölçer.

Bu, bir yayında bildirilecek sayı seçerken verdiğiniz kararı doğrudan etkiler.
Yeni kimyasal serilere genellemek istiyorsanız, iskelet bazlı sayıyı bildirmek
daha dürüsttür.

In [ ]:
karsilastirma = []

for bolunme_adi, (eg_idx, te_idx) in [
    ("Rastgele", (r_egitim, r_test)),
    ("Iskelet bazli", (i_egitim, i_test)),
]:
    Xe, ye = X_tam.iloc[eg_idx], y_hedef.iloc[eg_idx]
    Xt, yt = X_tam.iloc[te_idx], y_hedef.iloc[te_idx]

    pipeline, _ = secilen_uzay(optuna.trial.FixedTrial(calisma.best_params))
    cv_skor = cross_val_score(pipeline, Xe, ye, cv=kfold, scoring="r2", n_jobs=1).mean()
    pipeline.fit(Xe, ye)
    tahmin = pipeline.predict(Xt)

    karsilastirma.append({
        "Bolunme": bolunme_adi,
        "Egitim n": len(eg_idx),
        "Test n": len(te_idx),
        "CV R2 (Q2)": round(cv_skor, 4),
        "Test R2": round(r2_score(yt, tahmin), 4),
        "Test RMSE": round(float(np.sqrt(mean_squared_error(yt, tahmin))), 4),
        "Test MAE": round(mean_absolute_error(yt, tahmin), 4),
    })

df_karsilastirma = pd.DataFrame(karsilastirma)
print(f"Model: {secilen_ad}  |  Ayni hiperparametreler, farkli bolunme")
print("=" * 86)
print(df_karsilastirma.to_string(index=False))

dusus = (df_karsilastirma.loc[0, "Test R2"] - df_karsilastirma.loc[1, "Test R2"])
print(f"\nTest R2 dususu (rastgele -> iskelet bazli): {dusus:.4f}")
print("\nBu dusus, rastgele bolunmenin ne kadar iyimser oldugunu gosterir.")
print("Yeni kimyasal serilere genelleme hedefliyorsaniz, iskelet bazli sayiyi")
print("bildirmek daha durusttur.")

---
## 10. Sonuçları ve modeli kaydetme

Oturum 5'te (model karşılaştırma ve XAI) ve Oturum 6'da (yeni molekül tahmini)
kullanılacak her şeyi kaydediyoruz:

- **Sonuç tablosu** — bütün metrikler, Oturum 5'te sınıflandırma sonuçlarıyla
  birlikte tek tabloda toplanacak
- **Eğitilmiş model** (`joblib`) — Oturum 6'da yeni moleküller için tahmin
  üretecek. `Pipeline` kaydettiğimiz için ölçekleyici de modelle birlikte
  saklanır; Oturum 6'da ayrıca ölçekleme yapmak gerekmez.
- **Tahminler** — Oturum 5'teki uygulanabilirlik alanı analizi için

In [ ]:
# 1) Sonuc tablosu
sonuc_satirlari = [{
    "Gorev": "regresyon",
    "Model": secilen_ad,
    "Ozellik seti": OZELLIK_SETI,
    "Ozellik sayisi": X_tam.shape[1],
    "Bolunme": "rastgele",
    "Egitim n": len(egitim_idx),
    "Test n": len(test_idx),
    "CV R2 (Q2)": round(m_cv["R2"], 4),
    "Egitim R2": round(m_egitim["R2"], 4),
    "Test R2": round(m_test["R2"], 4),
    "Test RMSE": round(m_test["RMSE"], 4),
    "Test MAE": round(m_test["MAE"], 4),
    "y-random CV R2 (ort)": round(float(sahte_r2_cv.mean()), 4),
    "y-random CV R2 (max)": round(float(sahte_r2_cv.max()), 4),
}]
for satir in karsilastirma:
    if satir["Bolunme"] == "Iskelet bazli":
        sonuc_satirlari.append({
            "Gorev": "regresyon", "Model": secilen_ad,
            "Ozellik seti": OZELLIK_SETI, "Ozellik sayisi": X_tam.shape[1],
            "Bolunme": "iskelet bazli",
            "Egitim n": satir["Egitim n"], "Test n": satir["Test n"],
            "CV R2 (Q2)": satir["CV R2 (Q2)"], "Egitim R2": None,
            "Test R2": satir["Test R2"], "Test RMSE": satir["Test RMSE"],
            "Test MAE": satir["Test MAE"],
            "y-random CV R2 (ort)": None, "y-random CV R2 (max)": None,
        })

df_sonuc = pd.DataFrame(sonuc_satirlari)
df_sonuc.to_csv(BU / AD_SONUC, index=False)

# 2) LazyPredict tarama tablosu (Oturum 5 icin)
tarama_hepsi.to_csv(BU / "ntrk1_regresyon_lazypredict_tarama.csv")

# 3) Egitilmis model + ustveri
paket = {
    "pipeline": en_iyi_pipeline,
    "model_adi": secilen_ad,
    "ozellik_seti": OZELLIK_SETI,
    "ozellik_adlari": list(X_tam.columns),
    "hedef_kolon": HEDEF_KOLON,
    "en_iyi_parametreler": calisma.best_params,
    "cv_r2": m_cv["R2"],
    "test_r2": m_test["R2"],
    "test_rmse": m_test["RMSE"],
    "egitim_indeksleri": egitim_idx.tolist(),
    "test_indeksleri": test_idx.tolist(),
}
joblib.dump(paket, BU / AD_MODEL)

# 4) Test seti tahminleri (Oturum 5 icin)
pd.DataFrame({
    "Smiles": smiles.iloc[test_idx].values,
    f"gercek_{HEDEF_KOLON}": y_test.values,
    f"tahmin_{HEDEF_KOLON}": tahmin_test,
    "artik": artiklar,
}).to_csv(BU / "ntrk1_regresyon_test_tahminleri.csv", index=False)

print("Kaydedilen dosyalar")
print("-" * 70)
for ad in [AD_SONUC, "ntrk1_regresyon_lazypredict_tarama.csv", AD_MODEL,
           "ntrk1_regresyon_test_tahminleri.csv", AD_BOLUNME]:
    yol = BU / ad
    if yol.exists():
        print(f"  {ad:<48} {yol.stat().st_size / 1024:>8.1f} KB")
print(f"\nKlasor: {BU}")
print(df_sonuc.to_string(index=False))

---
## 11. Özet

### Ne yaptık?

1. Veriyi **iki farklı biçimde** böldük (rastgele ve iskelet bazlı) ve farkı
   ölçtük.
2. Eğitimden ayrı bir **doğrulama** seti kurup **LazyPredict** ile ~40 modeli
   taradık — test setine dokunmadan.
3. Üç **temsili** (betimleyici, ECFP4, MACCS) karşılaştırdık.
4. En iyi aileyi **Optuna** ile, 5 katlı çapraz doğrulama üzerinden ayarladık.
5. Son modeli eğitip test setinde **bir kez** değerlendirdik.
6. **y-randomizasyon** ile modelin gerçekten öğrendiğini doğruladık.
7. Artıkları inceledik ve her şeyi kaydettik.

### Akılda kalması gerekenler

- **Test seti kutsaldır.** Model seçimi ve ayarlaması bitmeden açılmaz.
  LazyPredict'e doğrulama setini verin, test setini değil.
- **Q² (CV R²), R²'den (eğitim) daha bilgilendiricidir.** Aradaki büyük fark
  aşırı öğrenme işaretidir.
- **Rastgele bölünme iyimserdir.** İskelet bazlı bölünme, yeni kimyasal
  serilere genellemeyi ölçer.
- **y-randomizasyon pazarlık konusu değildir.** Her QSAR modeli için
  bildirilmelidir.
- **Ölçekleme `Pipeline` içinde olur.** Böylece her CV katmanında yalnızca o
  katmanın eğitim verisine uydurulur.
- **LazyPredict bir eleme aracıdır, karar verici değil.** Varsayılan
  parametrelerle yapılan sıralama, ayarlamadan sonra değişebilir.

### Sırada ne var?

**Oturum 4 — Sınıflandırma.** Aynı veriyi, aynı protokolle, ama bu kez
"aktif mi değil mi?" sorusu için modelleyeceğiz. Yeni konular: sınıf
dengesizliği, ROC/PR eğrileri, karmaşıklık matrisi ve MCC.

---

### Alıştırmalar

1. `OZELLIK_SETI`'ni `"maccs"` yapıp not defterini baştan çalıştırın. 167 bit,
   yüzlerce betimleyici kadar iyi tahmin ediyor mu?
2. `ECFP_TARAMASI = True` yapıp ECFP4'ü de tarayın. En iyi temsil hangisi?
   Süre farkı ne kadar?
3. `OPTUNA_DENEME`'yi 100'e çıkarın. CV R² ne kadar iyileşiyor? Harcanan ek
   süreye değiyor mu?
4. Bölüm 9'daki karşılaştırmada iskelet bazlı bölünmeyle **baştan** optimizasyon
   yapın (yani Optuna'yı da iskelet bölünmesi üzerinde koşun). Seçilen
   hiperparametreler değişiyor mu?
5. Artık grafiğinde `|artık| > 1` olan molekülleri bulup yapılarını çizin.
   Ortak bir özellikleri var mı? (Bu, Oturum 5'teki uygulanabilirlik alanı
   tartışmasının hazırlığıdır.)